# 05. Історична таблиця курсів валют

Цей ноутбук відповідає за п’ятий етап конвеєра. Він читає з bronze курси за поточну дату та готує їх для історичної таблиці фактів.

Зерно таблиці `fact_exchange_rate` становить одна валюта за одну дату. Це означає, що пара `business_date` і `currency_key` не повинна повторюватися.

Ноутбук не звертається до API НБУ. Він читає дані з `nbu_raw.raw_rates` і використовує ключі валют із `nbu_dwh.dim_currency`.

Оскільки в BigQuery Sandbox команда `MERGE` недоступна, злиття виконуємо через pandas. Читаємо стару історію без поточної дати, додаємо актуальну порцію та повністю перезаписуємо таблицю.

## Послідовність роботи

1. Підключитися до BigQuery.
2. Прочитати з bronze дані за поточну дату.
3. Розгорнути JSON і нормалізувати значення.
4. Прибрати повтори всередині поточної порції.
5. Додати ключі дати та валюти.
6. Створити таблицю фактів, якщо її ще немає.
7. Об’єднати стару історію з актуальною порцією.
8. Записати результат у `nbu_dwh.fact_exchange_rate`.
9. Перевірити зерно та кількість рядків.

## 1. Налаштування та підключення

Вказуємо назву Google Cloud проєкту, імпортуємо потрібні бібліотеки та створюємо клієнт BigQuery.

Ноутбук підключається до BigQuery самостійно, оскільки оркестратор запускатиме кожен етап окремо.

Для авторизації використовуємо JSON-ключ сервісного акаунта зі змінної середовища `GCP_SA_KEY`. Сам ключ у коді не зберігається.

In [1]:
PROJECT_ID = "nbu-bigquery-etl"
LOCATION = "EU"

DS_RAW = "nbu_raw"
DS_DWH = "nbu_dwh"

RAW_TABLE = f"{PROJECT_ID}.{DS_RAW}.raw_rates"
DIM_CURRENCY_TABLE = f"{PROJECT_ID}.{DS_DWH}.dim_currency"
FACT_TABLE = f"{PROJECT_ID}.{DS_DWH}.fact_exchange_rate"

In [2]:
import os
import sys
import json

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.max_columns", 40)

In [3]:
creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account

    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]

    )

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)

print("проєкт:", client.project)

проєкт: nbu-bigquery-etl


## Завдання 6. Ноутбук 05: історична таблиця фактів

### Завдання 6.1. Читання даних за поточну дату

Читаємо з bronze тільки рядки, у яких `business_date` дорівнює поточній даті BigQuery.

На цьому етапі не завантажуємо всю історію bronze. Для щоденного оновлення таблиці фактів потрібна лише актуальна порція даних.

In [4]:
sql = f"""
SELECT ingested_at,
         business_date,
         payload
FROM `{RAW_TABLE}`
WHERE business_date = CURRENT_DATE()
"""

raw = client.query(sql).to_dataframe()

print(len(raw), "рядків прочитано з bronze")
raw.head(5)

90 рядків прочитано з bronze


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ingested_at,business_date,payload
0,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""DZD"", ""exchangedate"": ""24.08.2026"", ""r..."
1,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""AUD"", ""exchangedate"": ""24.08.2026"", ""r..."
2,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""BDT"", ""exchangedate"": ""24.08.2026"", ""r..."
3,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""CAD"", ""exchangedate"": ""24.08.2026"", ""r..."
4,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""CNY"", ""exchangedate"": ""24.08.2026"", ""r..."


### Завдання 6.2. Підготовка актуальної порції

Розгортаємо JSON із колонки `payload`, очищуємо код валюти та переводимо курс у числовий тип.

У bronze поточна порція може містити повтори через кілька запусків першого ноутбука. Тому для кожної пари `business_date` і `currency_code` залишаємо найсвіжіший рядок за `ingested_at`.

Після цього набір повинен відповідати зерну майбутньої таблиці фактів: одна валюта за одну дату.

In [5]:
parsed = pd.json_normalize(raw["payload"].map(json.loads))

print("колонки JSON:", list(parsed.columns))
print(len(parsed), "рядків після розгортання JSON")

parsed.head(5)

колонки JSON: ['cc', 'exchangedate', 'r030', 'rate', 'special', 'txt']
90 рядків після розгортання JSON


,cc,exchangedate,r030,rate,special,txt
0,DZD,24.08.2026,12,0.33613,NaN,Алжирський динар
1,AUD,24.08.2026,36,31.99060,NaN,Австралійський долар
2,BDT,24.08.2026,50,0.36509,NaN,Така
3,CAD,24.08.2026,124,32.50720,NaN,Канадський долар
4,CNY,24.08.2026,156,6.64470,NaN,Юань Женьміньбі


In [6]:
parsed["currency_code"] = parsed["cc"].str.strip().str.upper()
parsed["rate"] = pd.to_numeric(parsed["rate"], errors="coerce")

fact_data = parsed[["currency_code", "rate"]].copy()

df = pd.concat(
    [
        raw[["ingested_at", "business_date"]].reset_index(drop=True),
        fact_data.reset_index(drop=True)
    ],
    axis=1
)

print("колонки:", list(df.columns))
print(len(df), "рядків після нормалізації")

df.head(5)

колонки: ['ingested_at', 'business_date', 'currency_code', 'rate']
90 рядків після нормалізації


,ingested_at,business_date,currency_code,rate
0,2026-08-24 08:14:35+00:00,2026-08-24,DZD,0.33613
1,2026-08-24 08:14:35+00:00,2026-08-24,AUD,31.99060
2,2026-08-24 08:14:35+00:00,2026-08-24,BDT,0.36509
3,2026-08-24 08:14:35+00:00,2026-08-24,CAD,32.50720
4,2026-08-24 08:14:35+00:00,2026-08-24,CNY,6.64470


In [7]:
fact_latest  = (df.sort_values(["ingested_at"])
                .drop_duplicates(subset=["business_date", "currency_code"], keep="last")).reset_index(drop=True)

print(len(df), "рядків до дедуплікації")
print(len(fact_latest), "рядків після дедуплікації")

fact_latest.head(5)

90 рядків до дедуплікації
45 рядків після дедуплікації


,ingested_at,business_date,currency_code,rate
0,2026-08-24 08:15:17+00:00,2026-08-24,EGP,0.8779
1,2026-08-24 08:15:17+00:00,2026-08-24,TND,15.4032
2,2026-08-24 08:15:17+00:00,2026-08-24,AED,12.1587
3,2026-08-24 08:15:17+00:00,2026-08-24,ZAR,2.7883
4,2026-08-24 08:15:17+00:00,2026-08-24,CHF,55.8615


### Завдання 6.3. Додавання ключів вимірів

Створюємо `date_key` із `business_date` у форматі `YYYYMMDD`.

Ключ валюти не створюємо самостійно. Читаємо відповідність між `currency_code` і `currency_key` із таблиці `nbu_dwh.dim_currency` та приєднуємо її до актуальної порції.

Для кодів валют, яких немає у вимірі, використовуємо технічний ключ `-1`. Також додаємо `dw_load_ts`, який показує момент підготовки рядків для таблиці фактів.

In [8]:
sql = f"""
SELECT currency_key,
       currency_code
  FROM `{DIM_CURRENCY_TABLE}`
"""

dim_currency = client.query(sql).to_dataframe()

print(len(dim_currency), f"рядків прочитано з {DIM_CURRENCY_TABLE}")
dim_currency.head(5)

46 рядків прочитано з nbu-bigquery-etl.nbu_dwh.dim_currency


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,currency_key,currency_code
0,1,AED
1,2,AUD
2,3,AZN
3,4,BDT
4,5,CAD


In [9]:
fact = fact_latest.copy()

fact["date_key"] = (pd.to_datetime(fact["business_date"]).dt.strftime("%Y%m%d")).astype(int)

fact = fact.merge(dim_currency, how="left", on="currency_code", validate="one_to_one")

fact["currency_key"] = (fact["currency_key"].fillna(-1).astype(int))

fact["dw_load_ts"] = (pd.Timestamp.now(tz="UTC").floor("s"))

today_df = (fact[["date_key", "currency_key", "rate", "business_date", "dw_load_ts"]]
    .sort_values(["date_key", "currency_key"]).reset_index(drop=True))

print(len(today_df), "рядків у поточній порції")
today_df.head(5)

45 рядків у поточній порції


,date_key,currency_key,rate,business_date,dw_load_ts
0,20260824,1,12.15870,2026-08-24,2026-08-24 10:42:37+00:00
1,20260824,2,31.99060,2026-08-24,2026-08-24 10:42:37+00:00
2,20260824,3,26.27150,2026-08-24,2026-08-24 10:42:37+00:00
3,20260824,4,0.36509,2026-08-24,2026-08-24 10:42:37+00:00
4,20260824,5,32.50720,2026-08-24,2026-08-24 10:42:37+00:00


### Завдання 6.4. Створення таблиці фактів

Описуємо схему таблиці `nbu_dwh.fact_exchange_rate` та створюємо її, якщо вона ще не існує.

Таблицю партиціюємо за `business_date`. Це дозволяє BigQuery під час запитів за конкретну дату читати тільки потрібну частину даних.

Також додаємо кластеризацію за `currency_key`. Усередині кожної партиції рядки з однаковими ключами валют зберігатимуться поруч, що допомагає запитам із фільтрацією за валютою.

Використовуємо `exists_ok=True`, тому повторний запуск не завершиться помилкою, якщо таблиця вже створена.

In [10]:
fact_schema = [
    bigquery.SchemaField("date_key",      "INTEGER",   mode="REQUIRED"),
    bigquery.SchemaField("currency_key",  "INTEGER",   mode="REQUIRED"),
    bigquery.SchemaField("rate",          "FLOAT",     mode="REQUIRED"),
    bigquery.SchemaField("business_date", "DATE",      mode="REQUIRED"),
    bigquery.SchemaField("dw_load_ts",    "TIMESTAMP", mode="REQUIRED"),
]

table = bigquery.Table(FACT_TABLE, schema=fact_schema)

table.time_partitioning = bigquery.TimePartitioning(field="business_date")
table.clustering_fields = ["currency_key"]

client.create_table(table, exists_ok=True)
print("таблиця готова:", FACT_TABLE)

таблиця готова: nbu-bigquery-etl.nbu_dwh.fact_exchange_rate


### Завдання 6.5. Злиття історії через pandas

У безкоштовному BigQuery Sandbox команда `MERGE` недоступна, тому виконуємо злиття через pandas.

Спочатку читаємо з таблиці фактів усю історію, крім поточної дати. Потім додаємо підготовлену порцію `today_df` і повністю перезаписуємо таблицю через `WRITE_TRUNCATE`.

Таким способом попередня версія поточної дати замінюється новою, а рядки за інші дати зберігаються без змін.

In [11]:
# 1) уся історія, КРІМ сьогодні
old = client.query(f"""
SELECT date_key,
       currency_key,
       rate,
       business_date,
       dw_load_ts
  FROM `{FACT_TABLE}`
    WHERE business_date <> CURRENT_DATE()
""").to_dataframe()

print(len(old), "історичних рядків без поточної дати")

old.head(5)

0 історичних рядків без поточної дати


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_key,currency_key,rate,business_date,dw_load_ts


In [12]:
# 2) склеюємо з новою порцією
merged = pd.concat([old, today_df], ignore_index=True)

merged = (merged.sort_values(["date_key", "currency_key"]).reset_index(drop=True))

print("старих рядків:", len(old))
print("сьогоднішніх рядків:", len(today_df))
print("разом після злиття:", len(merged))

merged.head(5)

старих рядків: 0
сьогоднішніх рядків: 45
разом після злиття: 45


,date_key,currency_key,rate,business_date,dw_load_ts
0,20260824,1,12.15870,2026-08-24,2026-08-24 10:42:37+00:00
1,20260824,2,31.99060,2026-08-24,2026-08-24 10:42:37+00:00
2,20260824,3,26.27150,2026-08-24,2026-08-24 10:42:37+00:00
3,20260824,4,0.36509,2026-08-24,2026-08-24 10:42:37+00:00
4,20260824,5,32.50720,2026-08-24,2026-08-24 10:42:37+00:00


In [13]:
# 3) перезаписуємо таблицю цілком
cfg = bigquery.LoadJobConfig(schema=fact_schema, write_disposition="WRITE_TRUNCATE")

load_job = client.load_table_from_dataframe(merged, FACT_TABLE, job_config=cfg)

load_job.result()

table = client.get_table(FACT_TABLE)

print("таблиця:", FACT_TABLE)
print("записано рядків:", table.num_rows)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


таблиця: nbu-bigquery-etl.nbu_dwh.fact_exchange_rate
записано рядків: 45


### Завдання 6.6. Перевірка таблиці фактів

Читаємо записану таблицю `nbu_dwh.fact_exchange_rate` у DataFrame та виконуємо три перевірки за допомогою pandas.

Перевіряємо, що пара `date_key` і `currency_key` не повторюється. Потім рахуємо кількість різних `business_date` і загальну кількість рядків у таблиці.

Ці результати використаємо для перевірки повторного запуску ноутбука.

In [14]:
sql = f"""
SELECT date_key,
       currency_key,
       rate,
       business_date,
       dw_load_ts
  FROM `{FACT_TABLE}`
ORDER BY date_key,
      currency_key
"""

check_fact = client.query(sql).to_dataframe()

print(len(check_fact), f"рядків прочитано з {FACT_TABLE}")
check_fact.head(5)

45 рядків прочитано з nbu-bigquery-etl.nbu_dwh.fact_exchange_rate


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_key,currency_key,rate,business_date,dw_load_ts
0,20260824,1,12.15870,2026-08-24,2026-08-24 10:42:37+00:00
1,20260824,2,31.99060,2026-08-24,2026-08-24 10:42:37+00:00
2,20260824,3,26.27150,2026-08-24,2026-08-24 10:42:37+00:00
3,20260824,4,0.36509,2026-08-24,2026-08-24 10:42:37+00:00
4,20260824,5,32.50720,2026-08-24,2026-08-24 10:42:37+00:00


In [15]:
print("date_key + currency_key унікальні:", not check_fact.duplicated(subset=["date_key", "currency_key"]).any())

print("кількість різних business_date:", check_fact["business_date"].nunique())

print("загальна кількість рядків:", len(check_fact))

date_key + currency_key унікальні: True
кількість різних business_date: 1
загальна кількість рядків: 45


### Завдання 6.7. Перевірка повторного запуску

Я запустив ноутбук двічі підряд і порівняв результати перевірок.

Після першого запуску в таблиці була одна дата та 45 рядків. Після другого запуску кількість дат і загальна кількість рядків не змінилися.

Повторний запуск не створив дублікатів, оскільки перед злиттям ми читаємо стару історію без поточної дати, а потім додаємо її актуальну версію.

| Запуск | Кількість дат | Кількість рядків |
|---|---:|---:|
| Перший | 1 | 45 |
| Другий | 1 | 45 |

Це підтверджує, що злиття працює правильно та не накопичує повтори за поточну дату.

In [16]:
print("05_fact_merge завершено успішно")

05_fact_merge завершено успішно
